In [23]:
# import libraries
from ultralytics import YOLO
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input
import numpy as np
import cv2
import tensorflow as tf


def detect_and_predict(frame):
    results = yolo_model(frame)

    locs = []
    probs = []

    (h, w) = frame.shape[:2]

    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        for box, cls in zip(boxes, classes):

            # COCO class 16 = dog
            if int(cls) == 16:

                (x1, y1, x2, y2) = box.astype("int")

                # Clip bounds
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                dog_crop = frame[y1:y2, x1:x2]

                if dog_crop.size == 0:
                    continue

                # Preprocess
                dog_img = cv2.resize(dog_crop, (224, 224))
                dog_img = img_to_array(dog_img)
                dog_img = preprocess_input(dog_img)
                dog_img = np.expand_dims(dog_img, axis=0)

                # Convert to tensor
                dog_tensor = tf.convert_to_tensor(dog_img, dtype=tf.float32)

                # Prediction
                outputs = infer(dog_tensor)
                prob = float(next(iter(outputs.values())).numpy()[0][0])

                locs.append((x1, y1, x2, y2))
                probs.append(prob)

    return locs, probs

# Load the models
MODEL_PATH = r"D:\AI course\AI\Deep Learning\Dog_Behavior_Detection\dog_behavior_detector_eff_tfg"
VIDEO_PATH = r"D:\AI course\AI\Deep Learning\Dog_Behavior_Detection\dog.mp4"
yolo_model = YOLO("yolov8s.pt")
behavior_model = tf.saved_model.load(MODEL_PATH)
infer = behavior_model.signatures["serving_default"]

threshold = 0.5

# Initialize the video stream
print("[INFO] Starting video...")
vs = cv2.VideoCapture(VIDEO_PATH)

# Loop over frames
while True:
    ret, frame = vs.read()
    if not ret:
        break

    locs, probs = detect_and_predict(frame)

    for (box, prob) in zip(locs, probs):
        (x1, y1, x2, y2) = box
        if prob >= threshold:
            label = "Normal"
            confidence = prob
            color = (0, 255, 0)
        else:
            label = "Aggressive"
            confidence = 1 - prob
            color = (0, 0, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        text = f"{label}: {confidence*100:.1f}%"
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Show output
    cv2.imshow("Dog Behavior Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Cleanup
vs.release()
cv2.destroyAllWindows()

[INFO] Starting video...

0: 384x640 1 car, 2 dogs, 1 frisbee, 55.5ms
Speed: 31.3ms preprocess, 55.5ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 dogs, 1 frisbee, 53.5ms
Speed: 2.0ms preprocess, 53.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 dogs, 1 frisbee, 57.1ms
Speed: 2.2ms preprocess, 57.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 dogs, 1 frisbee, 50.4ms
Speed: 1.7ms preprocess, 50.4ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 dogs, 47.2ms
Speed: 1.5ms preprocess, 47.2ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 dogs, 48.4ms
Speed: 1.4ms preprocess, 48.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 dogs, 47.1ms
Speed: 1.6ms preprocess, 47.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1